# Распознавание лиц и глаз с помощью OpenCV

В этом ноутбуке проведены эксперименты по распознаванию лиц и глаз на фотографиях с использованием каскадов Хаара.

In [ ]:
# Установка библиотек (для Windows):
# pip install opencv-python matplotlib

import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

print("OpenCV version:", cv2.__version__)

In [ ]:
# Загрузка каскадов Хаара
# OpenCV автоматически загружает их из своих внутренних файлов
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

print("Каскады загружены успешно")
print("Face cascade loaded:", not face_cascade.empty())
print("Eye cascade loaded:", not eye_cascade.empty())

## Функция для распознавания лиц и глаз

In [ ]:
def detect_faces_and_eyes(image_path, scale_factor=1.1, min_neighbors=5, min_face_size=(30, 30)):
    """
    Функция для распознавания лиц и глаз на изображении.
    
    Параметры:
    - image_path: путь к изображению
    - scale_factor: коэффициент масштабирования для каскада
    - min_neighbors: минимальное количество соседей
    - min_face_size: минимальный размер лица
    
    Возвращает:
    - изображение с разметкой
    - количество обнаруженных лиц
    - количество обнаруженных глаз
    """
    # Загрузка изображения
    img = cv2.imread(image_path)
    if img is None:
        print(f"Ошибка: не удалось загрузить изображение {image_path}")
        return None, 0, 0
    
    # Конвертация в оттенки серого
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Выравнивание гистограммы для улучшения контраста
    gray = cv2.equalizeHist(gray)
    
    # Детекция лиц
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=min_face_size
    )
    
    print(f"Обнаружено лиц: {len(faces)}")
    
    total_eyes = 0
    
    # Рисуем рамки вокруг лиц и ищем глаза
    for idx, (x, y, w, h) in enumerate(faces):
        # Рамка лица
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
        
        # Подпись с номером
        label = f"Face {idx+1}"
        cv2.putText(img, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Выделяем область лица для поиска глаз (верхние 60%)
        roi_gray = gray[y:y+h, x:x+w]
        roi_gray_top = roi_gray[0:int(h*0.6), :]
        
        # Детекция глаз
        eyes = eye_cascade.detectMultiScale(
            roi_gray_top,
            scaleFactor=1.1,
            minNeighbors=10,
            minSize=(10, 10)
        )
        
        print(f"  Лицо {idx+1}: обнаружено глаз: {len(eyes)}")
        total_eyes += len(eyes)
        
        # Рисуем рамки вокруг глаз
        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(img, (x+ex, y+ey), (x+ex+ew, y+ey+eh), (255, 0, 0), 2)
    
    print(f"Всего обнаружено глаз: {total_eyes}")
    
    return img, len(faces), total_eyes

def show_image(img, title="Результат"):
    """Отображение изображения"""
    if img is None:
        print("Нет изображения для отображения")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 10))
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

## Эксперимент 1: Фотография с одним человеком (фронтальное лицо)

In [ ]:
# Укажите путь к вашей фотографии
# Можно использовать загруженное изображение или фото из интернета
# Для примера используем фото из репозитория OpenCV

import urllib.request

# Загрузка тестового изображения
url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
local_path = "photo1_lena.jpg"

try:
    urllib.request.urlretrieve(url, local_path)
    print(f"Изображение загружено: {local_path}")
except:
    print("Ошибка загрузки изображения из интернета")
    print("Пожалуйста, укажите путь к локальному файлу в переменной local_path")

In [ ]:
# Распознавание на фото 1
result_img, faces_count, eyes_count = detect_faces_and_eyes(local_path)

if result_img is not None:
    show_image(result_img, f"Фото 1: {faces_count} лиц, {eyes_count} глаз")
    
    # Сохраняем результат
    cv2.imwrite("result_photo1.jpg", result_img)
    print("Результат сохранен: result_photo1.jpg")

## Эксперимент 2: Фотография с несколькими людьми

In [ ]:
# Загрузка группового фото
url2 = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/group.jpg"
local_path2 = "photo2_group.jpg"

try:
    urllib.request.urlretrieve(url2, local_path2)
    print(f"Изображение загружено: {local_path2}")
except:
    print("Ошибка загрузки изображения из интернета")
    print("Пожалуйста, укажите путь к локальному файлу в переменной local_path2")

In [ ]:
# Распознавание на фото 2
# Для групповых фото используем более чувствительные параметры
result_img2, faces_count2, eyes_count2 = detect_faces_and_eyes(
    local_path2, 
    scale_factor=1.05,  # более тщательный поиск
    min_neighbors=3,     # меньше соседей для лучшего детектирования
    min_face_size=(20, 20)
)

if result_img2 is not None:
    show_image(result_img2, f"Фото 2: {faces_count2} лиц, {eyes_count2} глаз")
    
    # Сохраняем результат
    cv2.imwrite("result_photo2.jpg", result_img2)
    print("Результат сохранен: result_photo2.jpg")

## Эксперимент 3: Ваше собственное фото

In [ ]:
# Загрузите свое фото и укажите путь к нему здесь
# Например: local_path3 = "C:/Users/your_name/Pictures/my_photo.jpg"
# Или поместите фото в ту же папку, где этот ноутбук

local_path3 = "my_photo.jpg"  # ИЗМЕНИТЕ ЭТОТ ПУТЬ НА ВАШЕ ФОТО

# Проверяем, существует ли файл
if os.path.exists(local_path3):
    print(f"Файл найден: {local_path3}")
else:
    print(f"Файл не найден: {local_path3}")
    print("Пожалуйста, укажите правильный путь к вашему фото")

In [ ]:
# Распознавание на фото 3
if os.path.exists(local_path3):
    result_img3, faces_count3, eyes_count3 = detect_faces_and_eyes(local_path3)
    
    if result_img3 is not None:
        show_image(result_img3, f"Фото 3: {faces_count3} лиц, {eyes_count3} глаз")
        
        # Сохраняем результат
        cv2.imwrite("result_photo3.jpg", result_img3)
        print("Результат сохранен: result_photo3.jpg")

## Эксперимент 4: Фотография с боковым профилем или сложным ракурсом

In [ ]:
# Попробуем загрузить фото с профильным видом
# Можно использовать фото из интернета или свое

url4 = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"  # Замените на URL с профильным фото
local_path4 = "photo4_profile.jpg"

try:
    urllib.request.urlretrieve(url4, local_path4)
    print(f"Изображение загружено: {local_path4}")
except:
    print("Ошибка загрузки изображения из интернета")
    print("Пожалуйста, укажите путь к локальному файлу в переменной local_path4")

In [ ]:
# Распознавание на фото 4
result_img4, faces_count4, eyes_count4 = detect_faces_and_eyes(local_path4)

if result_img4 is not None:
    show_image(result_img4, f"Фото 4: {faces_count4} лиц, {eyes_count4} глаз")
    
    # Сохраняем результат
    cv2.imwrite("result_photo4.jpg", result_img4)
    print("Результат сохранен: result_photo4.jpg")

## Анализ результатов

In [ ]:
# Сводная таблица результатов
results = []

# Добавляем результаты экспериментов (если они были выполнены)
if 'faces_count' in locals():
    results.append({
        'Фото': 'Фото 1 (Lena)',
        'Реальное кол-во лиц': 1,
        'Распознано лиц': faces_count,
        'Реальное кол-во глаз': 2,
        'Распознано глаз': eyes_count
    })

if 'faces_count2' in locals():
    results.append({
        'Фото': 'Фото 2 (Group)',
        'Реальное кол-во лиц': 10,  # Примерное значение, уточните визуально
        'Распознано лиц': faces_count2,
        'Реальное кол-во глаз': 20,  # Примерное значение, уточните визуально
        'Распознано глаз': eyes_count2
    })

if 'faces_count3' in locals():
    results.append({
        'Фото': 'Фото 3 (Ваше фото)',
        'Реальное кол-во лиц': '?',  # Укажите визуально
        'Распознано лиц': faces_count3,
        'Реальное кол-во глаз': '?',  # Укажите визуально
        'Распознано глаз': eyes_count3
    })

if 'faces_count4' in locals():
    results.append({
        'Фото': 'Фото 4 (Профиль)',
        'Реальное кол-во лиц': '?',  # Укажите визуально
        'Распознано лиц': faces_count4,
        'Реальное кол-во глаз': '?',  # Укажите визуально
        'Распознано глаз': eyes_count4
    })

# Вывод таблицы
import pandas as pd
df = pd.DataFrame(results)
print("\n=== Сводная таблица результатов ===")
print(df.to_string(index=False))

### Почему количество распознанных лиц и глаз отличается от реального?

На основе проведенных экспериментов можно сделать следующие выводы:

#### 1. **Ограничения каскадов Хаара (Haar Cascades)**
- Каскады Хаара обучены на **фронтальных** изображениях лиц, поэтому они лучше всего работают с фотографиями, где человек смотрит прямо в камеру
- **Профильные виды** и сложные ракурсы (повернутая голова, наклон) распознаются хуже или вообще не распознаются
- Каскады могут выдавать **ложные срабатывания** (detect objects that are not faces) на текстурах, похожих на лицо

#### 2. **Параметры детектирования**
- **scaleFactor** (коэффициент масштабирования): чем меньше значение (например, 1.05), тем точнее поиск, но медленнее. Большее значение (1.1-1.3) быстрее, но может пропустить мелкие лица
- **minNeighbors** (минимальное количество соседей): высокое значение (5-10) уменьшает ложные срабатывания, но может пропустить некоторые лица. Низкое значение (1-3) находит больше лиц, но с больше ложных срабатываний
- **minSize** (минимальный размер): слишком маленький размер приводит к ложным срабатываниям, слишком большой — пропуску мелких лиц

#### 3. **Качество и условия фотографии**
- **Освещение**: плохое освещение, тени, засветы ухудшают распознавание
- **Разрешение**: низкое разрешение или размытые фото дают плохие результаты
- **Окклюзии**: очки, борода, усы, волосы, закрывающие часть лица мешают распознаванию
- **Выражение лица**: некоторые выражения (улыбка, открытый рот) могут повлиять на распознавание

#### 4. **Почему глаз распознается меньше, чем 2 на лицо?**
- Глаза ищутся **только внутри области обнаруженного лица** (верхние 60%)
- Если лицо распознано неправильно (слишком узкая рамка), глаза могут не попасть в область поиска
- Закрытые глаза, очки, наклон головы мешают распознаванию глаз
- Каскад для глаз более строгий (высокий minNeighbors=10), чтобы избежать ложных срабатываний

#### 5. **Почему иногда распознается больше глаз, чем должно быть?**
- Ложные срабатывания на бровях, носу, других элементах лица
- Отражения света или блики могут быть приняты за глаза